In [17]:
!pip install -q transformers datasets evaluate rouge_score

import torch
import numpy as np
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq
import evaluate

dataset = load_dataset("ILSUM/ILSUM-1.0", "English")

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def filter_data(example):
    return (
        len(example['Article'].split()) <= MAX_INPUT_LENGTH and
        len(example['Summary'].split()) <= MAX_TARGET_LENGTH
    )

dataset = dataset.filter(filter_data)

model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = examples["Article"]
    targets = examples["Summary"]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True)

    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    do_eval=True,   # instead of evaluation_strategy
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(600)),
    data_collator=data_collator
)

trainer.train()

trainer.save_model("/kaggle/working/bart_model")
tokenizer.save_pretrained("/kaggle/working/bart_model")

# -------- Manual Evaluation --------
rouge = evaluate.load("rouge")

test_samples = dataset["test"].select(range(50))

predictions = []
references = []

for example in test_samples:
    inputs = tokenizer(example["Article"], return_tensors="pt", truncation=True, max_length=512)

    summary_ids = model.generate(
        inputs["input_ids"].to(model.device),
        max_length=128,
        num_beams=4
    )

    pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    predictions.append(pred)
    references.append(example["Summary"])

results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
print(results)

# -------- Example Output --------
text = dataset["test"][0]["Article"]

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,1.535930
100,1.078045
150,0.912447
200,0.714073
250,0.671413
300,0.775680
350,0.572414
400,0.505154
450,0.700218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'rouge1': np.float64(0.415624416110986), 'rouge2': np.float64(0.2749725550146076), 'rougeL': np.float64(0.3617810839458373), 'rougeLsum': np.float64(0.36152275835427766)}


In [19]:
inputs["input_ids"] = inputs["input_ids"].to(model.device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=80,
    min_length=25,
    num_beams=6,
    length_penalty=3.0,
    no_repeat_ngram_size=3,
    repetition_penalty=1.3,
    early_stopping=True
)

print("\nOriginal:\n", text)
print("\nSummary:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))


Original:
 Indian-origin boy finds millions of years old fossil in UK gardenA six-year-old Indian-origin boy says he is “really excited” after he found a fossil from millions of years ago while digging in his garden in the West Midlands region of England. Siddak Singh Jhamat, known as Sid, was using a fossil-hunting set he received as a Christmas present when he came across a rock that looked like a horn."I was just digging for worms and things like pottery and bricks and I just came across this rock which looked a bit like a horn, and thought it could be a tooth or a claw or a horn, but it was actually a piece of coral which is called horn coral," the schoolboy said."I was really excited about what it really was," he said.According to a BBC report, his father Vish Singh was able to identify the horn coral through a fossil group he is a member of on Facebook and estimates the fossil is between 251 to 488 million years old."We were surprised he found something so odd-shaped in the soil